load delta tables

In [0]:
crm = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/crm")

billing = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/billing")

analytics = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/analytics")

join records

In [0]:
missing_in_billing = crm.join( billing,"customer_id","left_anti")

missing_in_billing.show()

+-----------+----------------+--------------------+-----------+-------------+
|customer_id|            name|               email|signup_date|         city|
+-----------+----------------+--------------------+-----------+-------------+
|  CRM002099|      Aadhya Rao|aadhya.rao716@hot...| 2024-02-09|   Coimbatore|
|  CRM007486|      Amit Kumar|amit.kumar838@yah...| 2023-05-22|        Patna|
|  CRM002073|    Ishaan Patel|ishaan.patel127@g...| 2024-01-05|        Surat|
|  CRM001054|     Suresh Bose|suresh.bose147@ya...| 2022-07-29|    Bangalore|
|  CRM007648|      Riya Mehta|riya.mehta622@yah...| 2024-02-17|Visakhapatnam|
|  CRM004693|    Ishaan Sinha|ishaan.sinha846@r...| 2023-02-14|        Surat|
|  CRM008927|    Vivaan Sinha|vivaan.sinha838@y...| 2023-12-15|       Mumbai|
|  CRM006797|    Pooja Sharma|pooja.sharma78@ya...| 2023-05-27|        Delhi|
|  CRM000073|Pooja Chatterjee|pooja.chatterjee2...| 2023-10-26|       Indore|
|  CRM003447|     Reyan Mehta|reyan.mehta261@ya...| 2024-03-28| 

Join records

In [0]:
missing_in_crm = billing.join( crm, "customer_id","left_anti")

missing_in_crm.show()

+-----------+--------------+-------+----------------+---------+
|customer_id|transaction_id| amount|transaction_date|   status|
+-----------+--------------+-------+----------------+---------+
| GHOST00659|    TXN0004538| 372.01|      2022-11-11|completed|
| GHOST00066|    TXN0011699|  19.78|      2023-11-13|completed|
| GHOST00705|    TXN0000971|  555.3|      2023-10-26|completed|
| GHOST00849|    TXN0006129|   47.8|      2023-11-15|completed|
| GHOST00692|    TXN0003353|  155.3|      2023-02-21|completed|
| GHOST00235|    TXN0006309| 407.73|      2024-05-02|completed|
| GHOST00579|    TXN0007642|  53.28|      2024-05-18|completed|
| GHOST00454|    TXN0008943|  83.65|      2022-04-14|completed|
| GHOST00367|    TXN0004572| 105.99|      2023-11-10|completed|
| GHOST00580|    TXN0011679|   7.33|      2023-09-18|  pending|
| GHOST00247|    TXN0010126| 791.27|      2022-01-31|completed|
| GHOST00819|    TXN0011317|  98.42|      2023-12-05|completed|
| GHOST01078|    TXN0003336| 276.35|    

In [0]:
from pyspark.sql.functions import count

crm_duplicates = crm.groupBy("customer_id") \
    .agg(count("*").alias("count")) \
    .filter("count>1")

crm_duplicates.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



join records

In [0]:
customer_match = crm.join( billing, "customer_id", "inner")

customer_match.show()

+-----------+-------------+--------------------+-----------+-------------+--------------+-------+----------------+---------+
|customer_id|         name|               email|signup_date|         city|transaction_id| amount|transaction_date|   status|
+-----------+-------------+--------------------+-----------+-------------+--------------+-------+----------------+---------+
|  CRM009746|     Sai Shah|sai.shah35@gmail.com| 2023-07-14|      XX_CITY|    TXN0000398| 227.58|      2023-05-01|completed|
|  CRM007145|  Rohan Mehta|rohan.mehta565@ya...| 2023-11-21|       Nagpur|    TXN0006282| 130.48|      2023-09-08|completed|
|  CRM008456|   Sneha Bose|sneha.bose955@red...| 2022-11-21|      Chennai|    TXN0011118|  644.3|      2023-12-02|completed|
|  CRM007689| Reyan Tiwari|                NULL| 2022-01-28|    Bangalore|    TXN0008497|   NULL|      2022-08-18|   failed|
|  CRM005422|   Aarav Shah|aarav.shah397@gma...| 2023-09-09|       Mumbai|    TXN0011399|   60.9|      2023-05-30|completed|


Saving data into delta table (gold)

In [0]:
missing_in_billing.write.format("delta").mode("overwrite").save("/Workspace/Cross-System Data Monitoring Platform/gold/missing_records")

crm_duplicates.write.format("delta").mode("overwrite").save("/Workspace/Cross-System Data Monitoring Platform/gold/duplicates")